# Lab 2 — Agent ReAct codé **à la main**
**Séance 2 · Du LLM à l'agent · PGE5 / M2**

On reconstruit la boucle d'un agent **sans framework**, pour comprendre ce qui se passe
réellement sous le capot de LangChain / CrewAI (Lab 3).

> **ReAct** = *Reasoning + Acting* (Yao et al., 2022) : le modèle **alterne** raisonnement,
> appel d'outil, observation du résultat, jusqu'à une réponse finale.

### Parcours (du simple au robuste)
1. La **boucle ReAct** explicite et instrumentée.
2. **Enchaînement** de plusieurs outils.
3. **Mémoire** : conversationnelle (dans la session) puis **long terme** (persistée sur disque).
4. **Sortie finale structurée** (`answer` + `sources` + `confidence`).
5. **Limites** : erreurs d'outil, hallucinations, boucles.
6. **Garde-fous** : anti-boucle, *allow-list*, « je ne sais pas ».
7. **Auto-critique** (Reflexion-lite) : l'agent relit et corrige sa réponse.

> Tout tourne **hors-ligne** (mock) sans clé d'API ; renseignez `.env` pour le vrai modèle.

## 0. Boîte à outils & mise en route

Quatre outils **locaux, déterministes et sûrs** (pas d'appel réseau → reproductible).
Le `calculator` utilise `safe_calc` (parsing **AST**, pas de `eval` !).

In [2]:
from llm_helpers import (
    make_client, credentials_available, ToolRegistry, tool_schema,
    run_agent, safe_calc,
)
import datetime, json

ONLINE = credentials_available()
print("Mode :", "🌐 EN LIGNE" if ONLINE else "⚙️  HORS-LIGNE (mock)")

def demo(script=None):
    return make_client(offline_script=script, quiet=True)

KB = {
    "craft ai": "Craft AI est une plateforme française/européenne de MLOps et LLMOps.",
    "react": "ReAct (Yao et al., 2022) combine raisonnement et action via appels d'outils.",
    "pge5": "PGE5 désigne la 5e année du Programme Grande École.",
    "reflexion": "Reflexion (Shinn et al., 2023) ajoute une auto-critique pour s'améliorer.",
}

def calculator(expression: str) -> str:
    "Évalue une expression arithmétique (sûr, sans eval)."
    return str(safe_calc(expression))

def knowledge_search(query: str) -> str:
    "Recherche naïve par mots-clés dans une base locale."
    q = query.lower()
    hits = [v for k, v in KB.items() if k in q or any(w in q for w in k.split())]
    return " ".join(hits) if hits else "Aucun résultat."

def today(_: str = "") -> str:
    "Donne la date du jour (ISO)."
    return datetime.date.today().isoformat()

def unit_convert(value: float, factor: float) -> str:
    "Convertit une valeur en la multipliant par un facteur."
    return f"{value * factor:.4g}"

registry = ToolRegistry()
registry.register(tool_schema("calculator", "Évalue une expression arithmétique.",
                              {"expression": {"type": "string"}}, ["expression"]), calculator)
registry.register(tool_schema("knowledge_search", "Cherche un fait dans la base locale.",
                              {"query": {"type": "string"}}, ["query"]), knowledge_search)
registry.register(tool_schema("today", "Donne la date du jour (ISO).", {}), today)
registry.register(tool_schema("unit_convert", "Convertit value*factor.",
                              {"value": {"type": "number"}, "factor": {"type": "number"}},
                              ["value", "factor"]), unit_convert)
print("Outils :", registry.names)

Mode : 🌐 EN LIGNE
Outils : ['calculator', 'knowledge_search', 'today', 'unit_convert']


## 1. La boucle ReAct, écrite explicitement et **instrumentée**

Plutôt que `run_agent`, on déroule la boucle nous-mêmes et on collecte une **trace**
(chaque appel d'outil + observation). C'est exactement ce qu'un framework masque.

In [3]:
def react_loop(client, registry, question, max_steps=6, system=None, verbose=True):
    "Boucle ReAct minimale. Renvoie (réponse_finale, trace)."
    system = system or ("Tu es un agent. Raisonne étape par étape et utilise les outils "
                        "pour répondre avec précision. Ne devine jamais un calcul : utilise calculator.")
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    trace = []
    for step in range(1, max_steps + 1):
        reply = client.complete(messages, tools=registry.specs)
        messages.append(reply.to_message())
        if not reply.has_tool_calls:
            if verbose: print(f"[{step}] FINAL : {reply.content}")
            return reply.content, trace
        for tc in reply.tool_calls:
            obs = registry.call(tc["name"], tc["arguments"])
            trace.append({"step": step, "tool": tc["name"], "args": tc["arguments"], "obs": obs})
            if verbose: print(f"[{step}] {tc['name']}({tc['arguments']}) -> {obs}")
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "name": tc["name"], "content": obs})
    return "⚠️ Limite d'étapes atteinte.", trace

client = demo([
    {"tool": "calculator", "arguments": {"expression": "1764**0.5"}},
    {"tool": "knowledge_search", "arguments": {"query": "craft ai"}},
    {"final": "√1764 = 42. Craft AI est une plateforme française/européenne de MLOps/LLMOps."},
])
answer, trace = react_loop(client, registry,
    "Quelle est la racine carrée de 1764, et qu'est-ce que Craft AI ?")
print("\nTrace :", json.dumps(trace, ensure_ascii=False, indent=2))

[1] calculator({'expression': '1764**(1/2)'}) -> 42.0
[1] knowledge_search({'query': 'Craft AI'}) -> Craft AI est une plateforme française/européenne de MLOps et LLMOps.
[2] FINAL : La racine carrée de 1764 est 42. Craft AI est une plateforme française/européenne de MLOps et LLMOps.

Trace : [
  {
    "step": 1,
    "tool": "calculator",
    "args": {
      "expression": "1764**(1/2)"
    },
    "obs": "42.0"
  },
  {
    "step": 1,
    "tool": "knowledge_search",
    "args": {
      "query": "Craft AI"
    },
    "obs": "Craft AI est une plateforme française/européenne de MLOps et LLMOps."
  }
]


## 2. Enchaînement de plusieurs outils

Une question dont la réponse **dépend** du résultat d'un premier outil. Observez comment
l'agent réutilise l'observation précédente.

In [4]:
client = demo([
    {"tool": "today", "arguments": {}},
    {"tool": "knowledge_search", "arguments": {"query": "reflexion"}},
    {"final": "Aujourd'hui (date ci-dessus), et Reflexion ajoute une auto-critique pour s'améliorer."},
])
react_loop(client, registry,
    "Quelle est la date du jour ? Et explique ce qu'apporte Reflexion.");

[1] today({}) -> 2026-06-22
[1] knowledge_search({'query': 'Reflexion'}) -> Reflexion (Shinn et al., 2023) ajoute une auto-critique pour s'améliorer.
[2] FINAL : La date du jour est le 22 juin 2026.

Reflexion, selon Shinn et al. (2023), est un mécanisme qui intègre l'auto-critique pour améliorer les performances d'un système. En d'autres termes, il permet au système de s'évaluer et de se corriger afin de progresser.


## 3. Mémoire

### 3.1 Mémoire **conversationnelle** (dans la session)
Un agent utile se souvient des tours précédents. On encapsule la boucle dans une classe
qui **conserve l'historique** entre les questions.

In [5]:
class ConversationAgent:
    "Agent ReAct qui garde l'historique de la conversation entre les tours."
    def __init__(self, client, registry, system="Tu es un assistant utile et concis."):
        self.client, self.registry = client, registry
        self.messages = [{"role": "system", "content": system}]

    def ask(self, user, max_steps=6):
        self.messages.append({"role": "user", "content": user})
        for _ in range(max_steps):
            reply = self.client.complete(self.messages, tools=self.registry.specs)
            self.messages.append(reply.to_message())
            if not reply.has_tool_calls:
                return reply.content
            for tc in reply.tool_calls:
                obs = self.registry.call(tc["name"], tc["arguments"])
                self.messages.append({"role": "tool", "tool_call_id": tc["id"],
                                      "name": tc["name"], "content": obs})
        return "⚠️ limite atteinte"

agent = ConversationAgent(demo([
    {"final": "Enchanté Marie ! Je note que tu révises les agents IA."},
    {"final": "Tu t'appelles Marie."},
]), registry)
print("T1 :", agent.ask("Bonjour, je m'appelle Marie et je révise les agents IA."))
print("T2 :", agent.ask("Au fait, quel est mon prénom ?"))   # nécessite la mémoire du tour 1

T1 : Bonjour Marie ! Je suis un agent IA, ravi de vous rencontrer. Comment puis-je vous aider aujourd'hui dans vos révisions sur les agents IA ?

T2 : Votre prénom est Marie.


### 3.2 Mémoire **long terme** (persistée sur disque)

La mémoire conversationnelle disparaît à la fin de la session. Pour qu'un agent se souvienne
**d'une exécution à l'autre**, on lui donne deux **outils** branchés sur un petit magasin JSON :
`remember(key, value)` et `recall(query)`.

In [6]:
import os
MEM_PATH = "agent_memory.json"

def _load_mem():
    return json.load(open(MEM_PATH, encoding="utf-8")) if os.path.exists(MEM_PATH) else {}

def _save_mem(d):
    json.dump(d, open(MEM_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

def remember(key: str, value: str) -> str:
    "Mémorise durablement une information (clé -> valeur)."
    d = _load_mem(); d[key] = value; _save_mem(d)
    return f"OK, mémorisé : {key} = {value}"

def recall(query: str) -> str:
    "Retrouve des informations mémorisées contenant la requête."
    d = _load_mem()
    hits = {k: v for k, v in d.items()
            if query.lower() in k.lower() or query.lower() in str(v).lower()}
    return json.dumps(hits, ensure_ascii=False) if hits else "Rien en mémoire."

mem_registry = ToolRegistry()
mem_registry.register(tool_schema("remember", "Mémorise une info durablement.",
                                  {"key": {"type": "string"}, "value": {"type": "string"}},
                                  ["key", "value"]), remember)
mem_registry.register(tool_schema("recall", "Retrouve une info mémorisée.",
                                  {"query": {"type": "string"}}, ["query"]), recall)

# --- Session 1 : l'agent mémorise ---
react_loop(demo([
    {"tool": "remember", "arguments": {"key": "projet_favori", "value": "un agent RAG sur Craft AI"}},
    {"final": "C'est noté : ton projet favori est un agent RAG sur Craft AI."},
]), mem_registry, "Retiens que mon projet favori est un agent RAG sur Craft AI.",
   system="Utilise remember/recall pour gérer la mémoire de l'utilisateur.");

[1] remember({'key': 'projet favori', 'value': 'un agent RAG sur Craft AI'}) -> OK, mémorisé : projet favori = un agent RAG sur Craft AI
[2] FINAL : D'accord, j'ai retenu que votre projet favori est un agent RAG sur Craft AI.


In [7]:
# --- Session 2 : un agent TOUT NEUF (aucun historique) retrouve l'info sur disque ---
react_loop(demo([
    {"tool": "recall", "arguments": {"query": "projet"}},
    {"final": "D'après ma mémoire, ton projet favori est un agent RAG sur Craft AI."},
]), mem_registry, "De quel projet je t'avais parlé la dernière fois ?",
   system="Utilise remember/recall pour gérer la mémoire de l'utilisateur.")
print("\nFichier de mémoire sur disque :", _load_mem())

[1] recall({'query': 'projet'}) -> {"projet favori": "un agent RAG sur Craft AI"}
[2] FINAL : La dernière fois, tu m'as parlé d'un agent RAG sur Craft AI comme ton projet favori.

Fichier de mémoire sur disque : {'projet favori': 'un agent RAG sur Craft AI'}


## 4. Sortie finale **structurée**

Pour brancher l'agent dans un système, sa réponse finale doit être exploitable : on demande
un JSON `{answer, sources, confidence}` plutôt que du texte libre. On laisse l'agent utiliser
ses outils, **puis** on force une dernière réponse structurée.

In [8]:
FINAL_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "sources": {"type": "array", "items": {"type": "string"},
                    "description": "outils/faits utilisés"},
        "confidence": {"type": "number", "description": "0 à 1"},
    },
    "required": ["answer", "confidence"],
}

def react_structured(client, registry, question, max_steps=5):
    messages = [{"role": "system", "content": "Agent rigoureux. Utilise les outils puis conclus."},
                {"role": "user", "content": question}]
    for _ in range(max_steps):
        reply = client.complete(messages, tools=registry.specs)
        messages.append(reply.to_message())
        if not reply.has_tool_calls:
            break
        for tc in reply.tool_calls:
            obs = registry.call(tc["name"], tc["arguments"])
            print(f"  · {tc['name']}({tc['arguments']}) -> {obs}")
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "name": tc["name"], "content": obs})
    messages.append({"role": "user", "content": "Donne ta réponse finale au format structuré."})
    return client.complete_structured(messages, FINAL_SCHEMA)

client = demo([
    {"tool": "calculator", "arguments": {"expression": "12*12"}},
    {"final": "Le calcul donne 144."},   # termine la boucle d'outils
    {"tool": "emit", "arguments": {"answer": "144", "sources": ["calculator"], "confidence": 0.99}},
])
print(react_structured(client, registry, "Combien font 12 fois 12 ?"))

  · calculator({'expression': '12 * 12'}) -> 144
{'confidence': 1, 'answer': '144'}


## 5. Observer les **limites**

### 5.1 Erreur d'outil — l'agent se rattrape-t-il ?
`calculator` lève une erreur sur `10/0` ; l'erreur est renvoyée au modèle (et non levée),
qui peut alors **changer de stratégie**.

In [9]:
client = demo([
    {"tool": "calculator", "arguments": {"expression": "10/0"}},   # va échouer
    {"tool": "today", "arguments": {}},
    {"final": "La division par zéro est indéfinie. Sinon, voici la date du jour ci-dessus."},
])
react_loop(client, registry, "Combien font 10 divisé par 0 ? Puis donne la date du jour.");

[1] calculator({'expression': '10 / 0'}) -> ERREUR pendant l'exécution de 'calculator': division by zero
[2] today({}) -> 2026-06-22
[3] FINAL : Il n'est pas possible de diviser un nombre par zéro. La division par zéro est indéfinie.

Nous sommes le 22 juin 2026.


### 5.2 Hallucination
Question **hors** base de connaissances. Un agent naïf invente ; un bon agent dit qu'il ne sait pas.

In [10]:
client = demo([
    {"tool": "knowledge_search", "arguments": {"query": "tournoi 2019"}},   # -> Aucun résultat
    {"final": "Je n'ai pas cette information dans ma base : je ne sais pas."},
])
react_loop(client, registry, "Qui a gagné le tournoi interne d'Aivancity en 2019 ?");

[1] knowledge_search({'query': "Qui a gagné le tournoi interne d'Aivancity en 2019 ?"}) -> Craft AI est une plateforme française/européenne de MLOps et LLMOps.
[2] FINAL : Je n'ai pas trouvé d'informations sur le gagnant du tournoi interne d'Aivancity en 2019 dans ma base de connaissances.


## 6. Garde-fous

On durcit la boucle : **(a)** *allow-list* d'outils autorisés, **(b)** **anti-boucle**
(refuser un appel **identique** déjà effectué), **(c)** budget d'étapes. Ces protections
sont indispensables avant toute mise en production.

In [ ]:
def guarded_react_loop(client, registry, question, allow=None, max_steps=6, system=None):
    system = system or "Agent prudent. Utilise les outils autorisés ; n'invente jamais."
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    seen = set()
    for step in range(1, max_steps + 1):
        reply = client.complete(messages, tools=registry.specs)
        messages.append(reply.to_message())
        if not reply.has_tool_calls:
            print(f"[{step}] FINAL : {reply.content}")
            return reply.content
        for tc in reply.tool_calls:
            name, args = tc["name"], tc["arguments"]
            sig = (name, json.dumps(args, sort_keys=True))
            if allow is not None and name not in allow:
                obs = f"REFUSÉ : '{name}' n'est pas dans l'allow-list {allow}."
            elif sig in seen:
                obs = "STOP : appel identique déjà effectué (anti-boucle)."
            else:
                seen.add(sig); obs = registry.call(name, args)
            print(f"[{step}] {name}({args}) -> {obs}")
            messages.append({"role": "tool", "tool_call_id": tc["id"], "name": name, "content": obs})
    return "⚠️ Budget d'étapes épuisé."

# Le mock répète volontairement le MÊME appel : l'anti-boucle l'intercepte au 2e coup.
client = demo([
    {"tool": "knowledge_search", "arguments": {"query": "pge5"}},
    {"tool": "knowledge_search", "arguments": {"query": "pge5"}},   # doublon -> bloqué
    {"final": "PGE5 = 5e année du Programme Grande École."},
])
guarded_react_loop(client, registry, "Qu'est-ce que PGE5 ?", allow={"knowledge_search", "today"});

## 7. Auto-critique — *Reflexion-lite*

Une amélioration clé : après une première réponse, l'agent **se relit**. Si la critique
détecte un problème, il **réessaie**. C'est l'idée de **Reflexion** (Shinn et al., 2023),
réduite à un tour.

In [ ]:
def react_with_reflection(client, registry, question):
    answer, _ = react_loop(client, registry, question, verbose=False)
    critique = client.complete([
        {"role": "system", "content": "Tu es un relecteur. Réponds 'OK' si la réponse est correcte "
                                       "et complète, sinon 'INSUFFISANT: <raison>'."},
        {"role": "user", "content": f"Question: {question}\nRéponse: {answer}"},
    ]).content
    print("Réponse initiale :", answer)
    print("Critique         :", critique)
    if critique and critique.strip().upper().startswith("INSUFFISANT"):
        answer2, _ = react_loop(client, registry,
            f"{question}\n(Améliore : {critique})", verbose=False)
        print("Réponse corrigée :", answer2)
        return answer2
    return answer

client = demo([
    {"final": "42."},                                   # 1re réponse, trop laconique
    {"final": "INSUFFISANT: la réponse n'explique pas le calcul."},  # critique
    {"final": "La racine carrée de 1764 vaut 42, car 42 × 42 = 1764."},  # correction
])
react_with_reflection(client, registry, "Quelle est la racine carrée de 1764 ? Explique.");

## 8. Exercices

> Tentez la cellule `# TODO`, puis dépliez la solution.

**8.1 — « Je ne sais pas » imposé.** Modifiez le *system prompt* de `guarded_react_loop`
pour que, si `knowledge_search` renvoie « Aucun résultat », l'agent réponde explicitement
« je ne sais pas » au lieu d'inventer. Testez sur une question hors base.

In [ ]:
# TODO : appeler guarded_react_loop avec un system imposant le "je ne sais pas".
# guarded_react_loop(demo([...]), registry, "Qui a inventé le café à Aivancity ?",
#                    allow={"knowledge_search"}, system="...")

<details><summary>✅ Solution 8.1</summary>

```python
SYS = ("Agent factuel. Si knowledge_search renvoie 'Aucun résultat', réponds exactement "
       "'Je ne sais pas.' et n'invente RIEN.")
guarded_react_loop(demo([
    {"tool": "knowledge_search", "arguments": {"query": "café Aivancity"}},
    {"final": "Je ne sais pas."},
]), registry, "Qui a inventé le café à Aivancity ?", allow={"knowledge_search"}, system=SYS)
```
</details>

**8.2 — Compteur de budget.** Ajoutez à `guarded_react_loop` un compteur d'appels d'outils
**total** et interrompez la boucle si l'agent dépasse `max_tool_calls=3` (différent de l'anti-boucle,
qui ne bloque que les doublons).

In [ ]:
# TODO : repartez de guarded_react_loop et ajoutez un budget d'appels d'outils.

<details><summary>✅ Solution 8.2</summary>

```python
def budgeted_loop(client, registry, question, max_tool_calls=3, max_steps=8):
    messages = [{"role": "system", "content": "Agent économe."},
                {"role": "user", "content": question}]
    used = 0
    for step in range(1, max_steps + 1):
        reply = client.complete(messages, tools=registry.specs)
        messages.append(reply.to_message())
        if not reply.has_tool_calls:
            return reply.content
        for tc in reply.tool_calls:
            if used >= max_tool_calls:
                return "⛔ Budget d'outils dépassé — arrêt."
            used += 1
            obs = registry.call(tc["name"], tc["arguments"])
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "name": tc["name"], "content": obs})
    return "⚠️ limite d'étapes."
```
</details>

**8.3 — Comparer les fournisseurs.** Renseignez `.env` avec OpenAI **puis** Anthropic
(ou Mistral) et relancez la section 1. Le même code, des comportements parfois différents :
notez les écarts (qualité des décisions d'outils, verbosité, respect du format).

## ✅ Récapitulatif

Vous avez codé un agent **complet, à la main** : boucle ReAct, mémoire (session + disque),
sortie structurée, garde-fous et auto-critique. Vous savez désormais **exactement** ce qu'un
framework automatise.

➡️ **Lab 3** : on réimplémente tout cela avec **LangGraph** (graphe d'états + *human-in-the-loop*),
on ajoute un **RAG**, puis un **système multi-agents** avec CrewAI.